In [3]:
import pandas as pd
import numpy as np
import spacy
from IPython.display import Image

In [4]:

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [5]:
!gdown 1sui9RXxVsPDa4s2kooQwRGhb8taZhcgD
!gdown 1H3gdo7SLBiWE_GGD6_xcdAp2wJJFcd5L

Downloading...
From: https://drive.google.com/uc?id=1sui9RXxVsPDa4s2kooQwRGhb8taZhcgD
To: c:\Users\VenkataKoushik\Desktop\Scaler\Lecture Notes\CV and NLP\Lecture_14\news_summary.csv

  0%|          | 0.00/11.9M [00:00<?, ?B/s]
  4%|▍         | 524k/11.9M [00:00<00:06, 1.81MB/s]
  9%|▉         | 1.05M/11.9M [00:00<00:05, 1.88MB/s]
 13%|█▎        | 1.57M/11.9M [00:00<00:06, 1.52MB/s]
 18%|█▊        | 2.10M/11.9M [00:01<00:08, 1.10MB/s]
 22%|██▏       | 2.62M/11.9M [00:03<00:14, 662kB/s] 
 26%|██▋       | 3.15M/11.9M [00:04<00:18, 462kB/s]
 31%|███       | 3.67M/11.9M [00:06<00:18, 453kB/s]
 35%|███▌      | 4.19M/11.9M [00:07<00:19, 389kB/s]
 40%|███▉      | 4.72M/11.9M [00:09<00:19, 368kB/s]
 44%|████▍     | 5.24M/11.9M [00:11<00:20, 330kB/s]
 48%|████▊     | 5.77M/11.9M [00:13<00:18, 325kB/s]
 53%|█████▎    | 6.29M/11.9M [00:14<00:15, 359kB/s]
 57%|█████▋    | 6.82M/11.9M [00:15<00:13, 370kB/s]
 62%|██████▏   | 7.34M/11.9M [00:17<00:13, 337kB/s]
 66%|██████▌   | 7.86M/11.9M [00:19<00:12

In [6]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

config = {'min_text_len':40,
          'max_text_len':60,
          'max_summary_len':30,
          'latent_dim' : 300,
          'embedding_dim' : 200}

In [7]:
from rouge_score import rouge_scorer

summary = pd.read_csv('news_summary.csv', encoding='iso-8859-1')
raw = pd.read_csv('news_summary_more.csv', encoding='iso-8859-1')

raw = raw.rename(columns = {'headlines':'summary'})
summary = summary[['headlines', 'text']].rename(columns={'headlines':'summary'})

# Concatenate the summary and the raw files
df = pd.concat([raw, summary]).reset_index(drop=True)

summary.shape, raw.shape

((4514, 2), (98401, 2))

In [8]:
import nltk
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to
[nltk_data]    |     C:\Users\VenkataKoushik\AppData\Roaming\nltk_data
[nltk_data]    |     ...
[nltk_data]    |   Unzipping corpora\abc.zip.
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     C:\Users\VenkataKoushik\AppData\Roaming\nltk_data
[nltk_data]    |     ...
[nltk_data]    |   Unzipping corpora\alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     C:\Users\VenkataKoushik\AppData\Roaming\nltk_data
[nltk_data]    |     ...
[nltk_data]    |   Unzipping taggers\averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     C:\Users\VenkataKoushik\AppData\Roaming\nltk_data
[nltk_data]    |     ...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers\averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru t

True

In [9]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Dense, Embedding, Input, InputLayer, RNN, SimpleRNN, LSTM, Bidirectional, TimeDistributed
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf

import string
from nltk.corpus import stopwords
stop_words = stopwords.words('english')

from sklearn.model_selection import train_test_split

import spacy
from time import time
import numpy as np
